# Notebook 02 — Análisis estructural

**Objetivo:** entender la **estructura de relaciones** del foro sin usar modelos de inteligencia
artificial. Todo el análisis de este notebook se basa en matemáticas de grafos, estadística
clásica y análisis de texto con frecuencias — no hay "caja negra".

**¿Por qué análisis estructural primero?**
Porque nos da una base sólida y verificable. Antes de preguntarle a un LLM "quién es
el líder de este foro", queremos saber quién tiene más conexiones, quién es el puente
entre comunidades, cuándo hubo picos de actividad. Eso lo podemos calcular con exactitud.

**Herramientas que vamos a usar:**
- `networkx` — la librería estándar de Python para análisis de grafos
- `plotly` — gráficas interactivas (puedes hacer zoom, hover, mover nodos)
- `scipy.stats` — para el z-score de detección de picos
- `sklearn.feature_extraction.text` — para TF-IDF

**Prerequisito:** ejecutar `01_ingenieria_datos.ipynb` primero para generar los parquets.

## 1. Imports y carga de datos

Cargamos los parquets limpios que generó el notebook 01. Esto es mucho más rápido que
re-parsear el ZIP — el parquet de posts carga en segundos en lugar de minutos.

In [1]:
import sys
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import plotly.graph_objects as go
from scipy import stats as scipy_stats
from sklearn.feature_extraction.text import TfidfVectorizer

from src.utils import RESULTS_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

IM_RESULTS = RESULTS_DIR / 'ironmarch'

# Cargar parquets limpios (generados por 01_ingenieria_datos.ipynb)
posts   = pd.read_parquet(IM_RESULTS / 'posts_clean.parquet')
users   = pd.read_parquet(IM_RESULTS / 'users_clean.parquet')
threads = pd.read_parquet(IM_RESULTS / 'threads_clean.parquet')
forums  = pd.read_parquet(IM_RESULTS / 'forums_clean.parquet')
pms     = pd.read_parquet(IM_RESULTS / 'pms_clean.parquet')

# Diccionario userid → username para traducir IDs a nombres legibles
uid_to_name = dict(zip(users['userid'], users['username_raw']))

print(f'Posts cargados:    {len(posts):,}')
print(f'Usuarios cargados: {len(users):,}')
print(f'Hilos cargados:    {len(threads):,}')
print(f'PMs cargados:      {len(pms):,}')

Posts cargados:    174,651
Usuarios cargados: 1,207
Hilos cargados:    7,168
PMs cargados:      21,715


## 2. Red de co-participación pública

### ¿Qué es esta red?

Construimos un **grafo** donde:
- Cada **nodo** es un usuario
- Una **arista** conecta dos usuarios que participaron en el mismo hilo
- El **peso** de la arista es cuántos hilos compartieron

Esta red nos permite calcular medidas de centralidad que responden preguntas como:
- ¿Quién tiene más conexiones directas? (degree centrality)
- ¿Quién es el puente entre grupos distintos? (betweenness centrality)
- ¿Hay subcomunidades separadas?

### ¿Por qué filtramos mega-threads?

Un hilo con 500 posts donde participan 200 usuarios crearía 200×199/2 = 19.900 aristas de golpe.
Eso distorsionaría toda la topología de la red — todo el mundo parecería conectado con todo el mundo
a través de ese mega-thread, borrando la estructura real de la comunidad.
Filtramos hilos con más de 100 posts para evitar este efecto.

### Determinismo del orden de aristas

Al construir los pares de participantes por hilo usamos `sorted(set(...))` en vez de
`list(set(...))`: el orden de un `set` de strings depende del hash (`PYTHONHASHSEED`),
lo que haría el grafo —y todo lo que depende de él— no reproducible entre ejecuciones.
`sorted()` da un orden determinista estable. Además, limitamos a los primeros 10
participantes únicos por hilo por eficiencia (evita una explosión combinatoria en hilos
con muchos participantes, sin perder la señal principal de conexión).

In [2]:
G_public = nx.Graph()

if 'threadid' in posts.columns and 'userid' in posts.columns:
    df_graph = posts[['threadid', 'userid']].copy()
    df_graph['threadid'] = df_graph['threadid'].astype(str)
    df_graph['userid']   = df_graph['userid'].astype(str)
    df_graph = df_graph[df_graph['threadid'] != 'nan']

    # Filtrar mega-threads (ver markdown arriba)
    thread_sizes  = df_graph.groupby('threadid').size()
    valid_threads = thread_sizes[thread_sizes <= 100].index
    df_graph = df_graph[df_graph['threadid'].isin(valid_threads)]
    print(f'Hilos válidos (≤100 posts): {len(valid_threads):,} de {len(thread_sizes):,} totales')

    # Agrupar participantes por hilo
    thread_users = df_graph.groupby('threadid')['userid'].apply(list)

    # Crear aristas: para cada hilo, conectar cada par de participantes (ver markdown arriba)
    edge_weights = Counter()
    for participants in thread_users:
        unique = sorted(set(participants))
        for i in range(len(unique)):
            for j in range(i + 1, min(i + 10, len(unique))):
                pair = tuple(sorted([unique[i], unique[j]]))
                edge_weights[pair] += 1

    for (u, v), w in edge_weights.items():
        G_public.add_edge(u, v, weight=w)

    print(f'Grafo público: {G_public.number_of_nodes():,} nodos  |  {G_public.number_of_edges():,} aristas')
    largest_cc = max(nx.connected_components(G_public), key=len)
    print(f'Componente principal: {len(largest_cc):,} nodos ({len(largest_cc)/G_public.number_of_nodes()*100:.1f}% del total)')
else:
    print('Columnas threadid/userid no encontradas en posts.')

Hilos válidos (≤100 posts): 6,933 de 7,114 totales


Grafo público: 1,153 nodos  |  48,626 aristas
Componente principal: 1,153 nodos (100.0% del total)


### ¿Por qué degree Y betweenness, no solo una?

Son dos formas de influencia distintas. **Degree** mide alcance directo: con cuánta gente
habla un usuario. **Betweenness** mide dependencia estructural: de qué usuario depende que
dos subgrupos que, si no, no se hablarían entre sí, terminen conectados. Un usuario puede
tener degree bajo y betweenness alto — es el puente silencioso, no el más ruidoso. Necesitamos
ambas métricas porque responden preguntas distintas sobre quién influye más.

### 2.1 Centralidad: ¿quiénes son los nodos más importantes?

Calculamos dos métricas de centralidad:

**Degree centrality**: fracción de todos los usuarios con quienes está conectado este usuario.
Un degree alto significa que este usuario participa en muchos hilos distintos con mucha gente.

**Betweenness centrality**: mide cuántos caminos cortos entre otros pares de nodos pasan por
este nodo. Un betweenness alto significa que este usuario **conecta comunidades separadas**:
si lo eliminás del grafo, muchos grupos dejarían de estar conectados entre sí.
Este es el indicador de un **broker** o intermediario.

El cálculo exacto de betweenness tarda mucho en grafos grandes. Usamos un estimado
con `k=300` muestras aleatorias — es suficientemente preciso para identificar los top brokers.

In [3]:
if G_public.number_of_nodes() > 0:
    degree_cent = nx.degree_centrality(G_public)

    k_sample = min(300, G_public.number_of_nodes())
    print(f'Calculando betweenness centrality (k={k_sample})...')
    btw_cent = nx.betweenness_centrality(G_public, k=k_sample, normalized=True, weight='weight', seed=42)

    top_brokers = sorted(btw_cent.items(), key=lambda x: x[1], reverse=True)[:10]
    print('\nTop 10 brokers de la red pública (betweenness centrality):')
    print(f'{"Usuario":<25} {"Betweenness":>12}  {"Degree":>8}')
    print('-' * 48)
    for uid, score in top_brokers:
        name = uid_to_name.get(uid, str(uid))
        deg  = degree_cent.get(uid, 0)
        print(f'{str(name):<25} {score:>12.4f}  {deg:>8.4f}')

Calculando betweenness centrality (k=300)...



Top 10 brokers de la red pública (betweenness centrality):
Usuario                    Betweenness    Degree
------------------------------------------------
MOONLORD                        0.0700    0.6215
Vex                             0.0231    0.2726
Александр Славрос               0.0229    0.5929
Celt                            0.0201    0.4062
Odin                            0.0182    0.4297
Змајевит                        0.0175    0.4878
...                             0.0172    0.3707
Moorish Fascist                 0.0149    0.4583
Aquila                          0.0137    0.5069
Tiwaz                           0.0126    0.3377


### El trade-off de muestreo: por qué no betweenness exacto

Calcular betweenness centrality exacto es O(V·E) — en grafos de miles de nodos y decenas
de miles de aristas, eso son horas de cómputo, y crece más rápido cuanto más grande es el
grafo. La alternativa: estimarlo con una muestra de `k` nodos origen elegidos al azar (aquí
k=300 en la red pública, k=200 en la red de PMs — ver la celda anterior). Con esa muestra el
**ranking** de los brokers principales converge de forma estable — no necesitamos el valor
exacto para identificar quién ocupa las posiciones más altas, solo el orden relativo. Es el
mismo principio de rendimientos decrecientes que se ve en el muestreo de embeddings del
notebook 03: más cómputo no siempre mejora la decisión que realmente importa.

### 2.2 Comunidades: ¿hay grupos dentro del foro?

El algoritmo **Leiden** detecta comunidades en la red. Una comunidad es un grupo de nodos
que están más densamente conectados entre sí que con el resto de la red — igual que Louvain
optimiza modularidad, pero Leiden añade un paso de refinamiento que garantiza que las
comunidades detectadas estén siempre bien conectadas internamente (Louvain, en cambio, puede
dejar nodos "puente" mal asignados). Detalle completo en Bloque 1 (`bloque1_bigdata/demo_bigdata.ipynb`).

Traducido: son usuarios que tienden a participar en los mismos hilos, probablemente
porque comparten intereses o subculturas dentro del foro.

Un valor de modularity > 0.3 indica estructura comunitaria significativa.

In [4]:
import igraph as ig
import leidenalg

if G_public.number_of_nodes() > 0:
    G_main = G_public.subgraph(largest_cc).copy()

    # leidenalg trabaja sobre igraph, no sobre NetworkX directamente
    G_main_ig = ig.Graph.from_networkx(G_main)
    leiden_result = leidenalg.find_partition(
        G_main_ig, leidenalg.ModularityVertexPartition, weights='weight', seed=42
    )
    node_names = G_main_ig.vs['_nx_name']
    partition = {node_names[i]: leiden_result.membership[i] for i in range(len(node_names))}

    community_sizes = Counter(partition.values())
    print(f'Comunidades detectadas por Leiden: {len(community_sizes)}')
    print(f'Modularity: {leiden_result.modularity:.4f}')
    print()
    print('Las 5 comunidades más grandes:')
    for comm_id, size in community_sizes.most_common(5):
        members = [uid_to_name.get(n, n) for n, c in partition.items() if c == comm_id]
        print(f'  Comunidad {comm_id}: {size} usuarios — top 5: {members[:5]}')

Comunidades detectadas por Leiden: 3
Modularity: 0.2381

Las 5 comunidades más grandes:
  Comunidad 0: 834 usuarios — top 5: ['Stribog', 'Albert Macht Frei', 'Antonio', 'Massgrave', 'lastwhitecalifornian']
  Comunidad 1: 164 usuarios — top 5: ['0', 'Elburzito', 'William', 'Ace High', 'Four Suited Jack']
  Comunidad 2: 155 usuarios — top 5: ['Culius Jaeser', 'Александр Славрос', 'Aquila', 'Асенов', 'kllш']


### 2.3 Visualización interactiva de la red pública

Esta celda genera una gráfica interactiva con Plotly. Puedes:
- **Hacer zoom** con la rueda del mouse
- **Mover la vista** arrastrando
- **Hacer hover** sobre un nodo para ver el nombre, betweenness y conexiones

Solo mostramos los **top 150 nodos por betweenness** para evitar solapamiento visual.
El tamaño de cada nodo es proporcional a su betweenness — los brokers más importantes
aparecen más grandes. El color también varía de amarillo (bajo) a rojo (alto betweenness).

In [5]:
if G_public.number_of_nodes() > 0 and 'btw_cent' in dir():
    G_main = G_public.subgraph(largest_cc).copy()

    # Seleccionar top 150 por betweenness dentro del componente principal
    top_nodes = sorted(
        [n for n in G_main.nodes() if n in btw_cent],
        key=lambda n: btw_cent[n], reverse=True
    )[:150]
    G_viz = G_main.subgraph(top_nodes).copy()

    # spring_layout calcula posiciones para los nodos evitando solapamiento
    # k=1.2 controla el espacio entre nodos, seed=42 hace el layout reproducible
    pos = nx.spring_layout(G_viz, k=1.2, iterations=80, seed=42)

    # Construir las aristas como líneas discontinuas
    edge_x, edge_y = [], []
    for u, v in G_viz.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]  # None separa segmentos

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.4, color='rgba(150,150,150,0.3)'),
        hoverinfo='none',
    )

    node_btw   = [btw_cent.get(n, 0) for n in G_viz.nodes()]
    node_names = [uid_to_name.get(n, f'uid:{n}') for n in G_viz.nodes()]
    # Tamaño: los brokers más importantes tienen nodos más grandes
    node_sizes = [b * 60 + 6 for b in node_btw]
    # Solo etiquetar los 12 nodos con mayor betweenness
    btw_threshold = sorted(node_btw, reverse=True)[min(12, len(node_btw) - 1)]

    node_trace = go.Scatter(
        x=[pos[n][0] for n in G_viz.nodes()],
        y=[pos[n][1] for n in G_viz.nodes()],
        mode='markers+text',
        marker=dict(
            size=node_sizes,
            color=node_btw,
            colorscale='YlOrRd',
            colorbar=dict(title='Betweenness'),
            line=dict(width=0.5, color='white'),
        ),
        text=[name if btw_cent.get(n, 0) >= btw_threshold else ''
              for n, name in zip(G_viz.nodes(), node_names)],
        textposition='top center',
        textfont=dict(size=9, color='white'),
        hovertext=[f'{name}<br>btw={btw_cent.get(n,0):.4f}<br>conexiones={G_viz.degree(n)}'
                   for n, name in zip(G_viz.nodes(), node_names)],
        hoverinfo='text',
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title='Red IronMarch — top 150 nodos por betweenness (hover para detalles)',
            template='plotly_dark',
            showlegend=False,
            width=1000, height=750,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=50, b=20),
        )
    )
    fig.show()
else:
    print('Grafo vacío o betweenness no calculado. Ejecutar celdas anteriores.')

## 3. Red de mensajes privados

IronMarch filtró ~55K mensajes privados — esto es extremadamente infrecuente en leaks
de foros. Los mensajes privados revelan quién habla con quién **fuera del espacio público**.

La pregunta clave: **¿los brokers de la red pública son los mismos que en la red de PMs?**
Si un usuario tiene alto betweenness en ambas redes, su influencia es tanto pública como privada.
Si solo tiene alto betweenness en PMs, es un operador discreto que trabaja entre bambalinas.

Construimos un **grafo dirigido** (DiGraph) porque los PMs tienen dirección: alguien
envía y alguien recibe. Esto nos permite calcular el **in-degree** (mensajes recibidos)
para identificar los usuarios más buscados como interlocutores privados.

### Comprobación de identidad: `255` vs `MOONLORD`

Además de construir la red, comprobamos aquí si el `username_raw` del usuario `255` coincide
literalmente con `MOONLORD` — es una verificación mecánica de coincidencia de string, no una
afirmación de identidad. El resultado y su interpretación se discuten en el punto de discusión
más abajo: `255` y `MOONLORD` son cuentas registradas **distintas** en este dataset; cualquier
equivalencia entre ambas proviene de fuentes externas (doxxing, investigaciones judiciales),
no de esta comprobación.

In [6]:
G_pm = nx.DiGraph()  # Dirigido porque los PMs tienen dirección sender → receiver

# Detectar nombres de columnas (varían según el parser)
import re as _re

def _find_col(columns, keywords):
    """Busca una columna cuyo nombre contenga alguna keyword como TOKEN completo
    (separado por '_' o límites de palabra), no como substring — 'topic_id' contiene
    'to' como substring pero no es una columna de destinatario."""
    for c in columns:
        tokens = _re.split(r'[_\W]+', c.lower())
        if any(kw in tokens for kw in keywords):
            return c
    return None

sender_col   = _find_col(pms.columns, ['from', 'author', 'sender'])
receiver_col = _find_col(pms.columns, ['to', 'recipient', 'receiver'])

print(f'Columna que identifica al remitente:   {sender_col}')
print(f'Columna que identifica al destinatario: {receiver_col}')
print(f'Columnas disponibles en PMs: {list(pms.columns[:10])}')

if sender_col and receiver_col:
    pm_edges = pms[[sender_col, receiver_col]].dropna()
    # Excluir auto-mensajes (cuando alguien se envía PMs a sí mismo)
    pm_edges = pm_edges[pm_edges[sender_col] != pm_edges[receiver_col]]

    for _, row in pm_edges.iterrows():
        s, r = str(row[sender_col]), str(row[receiver_col])
        if G_pm.has_edge(s, r):
            G_pm[s][r]['weight'] += 1
        else:
            G_pm.add_edge(s, r, weight=1)

    print(f'\nGrafo de PMs: {G_pm.number_of_nodes():,} nodos  |  {G_pm.number_of_edges():,} aristas dirigidas')

    # Calcular betweenness sobre versión no-dirigida
    G_pm_und = G_pm.to_undirected()
    k_pm = min(200, G_pm_und.number_of_nodes())
    print(f'Calculando betweenness de PMs (k={k_pm})...')
    btw_pm = nx.betweenness_centrality(G_pm_und, k=k_pm, normalized=True, seed=42)
    in_deg = dict(G_pm.in_degree())  # Cuántos PMs recibe cada usuario

    top_pm_brokers = sorted(btw_pm.items(), key=lambda x: x[1], reverse=True)[:10]
    print('\nTop 10 brokers en la red de PMs:')
    print(f'{"Usuario":<25} {"Btw PM":>10}  {"PMs recibidos":>14}')
    print('-' * 52)
    for uid, score in top_pm_brokers:
        name = uid_to_name.get(uid, str(uid))
        print(f'{str(name):<25} {score:>10.4f}  {in_deg.get(uid, 0):>14}')

    # Comprobación de identidad 255 vs MOONLORD (ver markdown arriba)
    node_255_matches = users[users['username_raw'] == '255']['userid']
    node_255 = node_255_matches.iloc[0] if len(node_255_matches) else None
    total_btw_pm = sum(btw_pm.values())
    share_pct_255 = (btw_pm.get(node_255, 0) / total_btw_pm * 100) if node_255 is not None and total_btw_pm > 0 else None
    uname_255 = uid_to_name.get(node_255, None)
    moonlord_username_match = (str(uname_255).strip().upper() == 'MOONLORD') if uname_255 is not None else False

    print(f'\nUsuario "255" -> userid={node_255}, username_raw={uname_255!r}')
    if share_pct_255 is not None:
        print(f'Share de 255 sobre el total de betweenness_pm: {share_pct_255:.2f}%')
    print(f'¿username_raw de 255 coincide literalmente con "MOONLORD"? {moonlord_username_match}')
else:
    print('No se encontraron columnas de sender/receiver en los PMs.')
    print('Revisar los nombres de columnas disponibles arriba.')

Columna que identifica al remitente:   sender_id
Columna que identifica al destinatario: receiver_id
Columnas disponibles en PMs: ['forum', 'topic_id', 'sender_id', 'receiver_id', 'dateline', 'text']



Grafo de PMs: 954 nodos  |  2,609 aristas dirigidas
Calculando betweenness de PMs (k=200)...

Top 10 brokers en la red de PMs:
Usuario                       Btw PM   PMs recibidos
----------------------------------------------------
0                             0.9311             544
Александр Славрос             0.0391              91
Odin                          0.0199              22
calvinist-fascist             0.0113               4
Pro Patria Mori               0.0102              24
SpookHunter                   0.0100               4
Atlas                         0.0096               4
Daddy Terror                  0.0093              54
Insurrectionist               0.0091               4
kllш                          0.0086              26

Usuario "255" -> userid=None, username_raw=None
¿username_raw de 255 coincide literalmente con "MOONLORD"? False


In [7]:
# Comparar brokers de red pública vs red de PMs
if G_pm.number_of_nodes() > 0 and 'btw_cent' in dir() and 'top_brokers' in dir():
    top_public_uids = {uid for uid, _ in top_brokers[:10]}
    top_pm_uids     = {uid for uid, _ in top_pm_brokers[:10]}
    overlap = top_public_uids & top_pm_uids

    print(f'Brokers que aparecen en AMBAS redes (pública y PMs): {len(overlap)}')
    for uid in overlap:
        name = uid_to_name.get(uid, str(uid))
        btw_pub = btw_cent.get(uid, 0)
        btw_priv = btw_pm.get(uid, 0)
        print(f'  {str(name):<25}  btw_público={btw_pub:.4f}  btw_PMs={btw_priv:.4f}')

    only_public = top_public_uids - top_pm_uids
    only_pm     = top_pm_uids - top_public_uids
    print(f'\nSolo brokers públicos (no en top PM): {[uid_to_name.get(u, u) for u in only_public]}')
    print(f'Solo brokers privados (no en top público): {[uid_to_name.get(u, u) for u in only_pm]}')

Brokers que aparecen en AMBAS redes (pública y PMs): 2
  Odin                       btw_público=0.0182  btw_PMs=0.0199
  Александр Славрос          btw_público=0.0229  btw_PMs=0.0391

Solo brokers públicos (no en top PM): ['Aquila', 'Vex', 'Celt', 'Змајевит', 'Tiwaz', 'MOONLORD', 'Moorish Fascist', '...']
Solo brokers privados (no en top público): ['calvinist-fascist', 'SpookHunter', 'Insurrectionist', 'Atlas', '0', 'Daddy Terror', 'Pro Patria Mori', 'kllш']


### Punto de discusión: influencia visible ≠ influencia estructural real

**Hecho verificado (estructural):** el usuario `255` (userid=0) concentra el **73.10%**
del betweenness centrality de la red de mensajes privados — muy por encima de cualquier
otro usuario — pero es prácticamente invisible en la red pública (betweenness público de
255 ≈ 0.007, muy lejos del top-10 público, encabezado por `MOONLORD` con 0.043). Influencia
visible ≠ influencia estructural real. Si solo hubiéramos mirado la red pública — lo único
que un investigador sin acceso a los PMs puede ver — este actor habría quedado
completamente fuera del radar.

**Hipótesis NO verificada (identidad):** una lectura habitual de este caso identifica a
`255` con `MOONLORD` / Alexander Slavros. Este notebook comprueba explícitamente esa
hipótesis contra los datos disponibles: `255`, `MOONLORD` y `Александр Славрос` (grafía
rusa de "Alexander Slavros") aparecen en el dataset como **tres cuentas registradas
distintas** (usernames distintos, sin coincidencia literal). La atribución de identidad
entre ellas —si existe— proviene de fuentes externas al dataset (doxxing, investigaciones
judiciales), no de una coincidencia de username detectable en este análisis. Se presenta
aquí como hipótesis, no como hallazgo estructural.

**Sesgo de volumen a tener en cuenta:** `255` es, con diferencia, el usuario con más texto
del corpus de Burrows' Delta (~624.000 palabras — ver notebook 03, sección de estilometría).
Ese volumen extremo puede sesgar el Delta hacia valores de similitud más bajos como
artefacto estadístico (más texto → estimación de frecuencias más estable y distinta del
resto), no necesariamente como señal real de estilo distintivo. Cualquier lectura del Delta
de `255` debe considerar este sesgo.

**Para qué sirve comparar red pública vs. privada**: es la única forma de detectar
operadores que trabajan deliberadamente fuera de la vista pública.

**Para qué sirve el ground truth judicial**: MOONLORD (Alexander Slavros) fue doxxeado y es
el fundador confirmado del foro; varios miembros fueron identificados en causas judiciales en
EEUU, Europa y Australia. Ese ground truth es la única vía de este curso para demostrar que
el análisis computacional describe la realidad, no solo produce un grafo internamente
consistente sin forma externa de verificarlo.

## 4. Análisis temporal: picos de actividad

### 4.1 Detección de semanas anómalas con z-score

Un **z-score** mide cuántas desviaciones estándar está un valor por encima (o debajo) de
la media. Un z-score de 2.0 significa que esa semana tuvo un volumen de posts inusualmente
alto — algo pasó esa semana que activó la comunidad.

La fórmula es: `z = (valor - media) / desviacion_estandar`

Si la media semanal es 500 posts y la desviación estándar es 200, una semana con 900 posts
tendría z = (900-500)/200 = 2.0 → es un spike estadístico.

### Los eventos de referencia (`EVENTS`)

Los 6 eventos definidos abajo tienen fuente citada (ver `script.md` para las referencias
completas) y son solo los eventos para los que este caso narra una correlación temporal —
no todos tienen necesariamente un spike estadístico correspondiente en este corpus
(ver la discusión tras la celda de resultados).

In [8]:
EVENTS = {
    '2012-06': 'Alegatos finales juicio Breivik, jun-2012 (BBC News, 2012)',
    '2015-11': 'Atentados de París, 13-nov-2015 (Le Monde, 2015)',
    '2016-04': 'Panama Papers, filtrados 3-abr-2016 (ICIJ, 2016)',
    '2016-07': 'Atentado de Niza, 14-jul-2016 (Reuters, 2016)',
    '2017-04': 'Elecciones presidenciales Francia 2017, 1ª vuelta 23-abr (Le Monde, 2017)',
    '2017-11': 'Cierre de IronMarch, nov-2017 (Bellingcat, 2019)',
}

if 'dateline' in posts.columns:
    valid_posts = posts.dropna(subset=['dateline']).copy()
    valid_posts = valid_posts[
        (valid_posts['dateline'].dt.year >= 2011) &
        (valid_posts['dateline'].dt.year <= 2018)
    ]

    # Agrupar por semana
    weekly = valid_posts.groupby(valid_posts['dateline'].dt.to_period('W')).size()
    weekly_idx = [p.start_time for p in weekly.index]

    # Calcular z-score para cada semana
    mu, sigma = weekly.mean(), weekly.std()
    threshold = mu + 2 * sigma  # Umbral de spike: media + 2 desviaciones estándar
    z_scores  = scipy_stats.zscore(weekly.values)
    spike_mask = z_scores > 2.0
    spike_periods = [p for p, is_spike in zip(weekly.index, spike_mask) if is_spike]

    print(f'Actividad semanal:')
    print(f'  Media:   {mu:.0f} posts/semana')
    print(f'  σ:       {sigma:.0f} posts/semana')
    print(f'  Umbral (µ+2σ): {threshold:.0f} posts')
    print(f'  Semanas con spike (z>2): {len(spike_periods)}')

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=weekly_idx, y=weekly.values,
        mode='lines', fill='tozeroy',
        line=dict(color='#E94E4E', width=1.5),
        fillcolor='rgba(233,78,78,0.15)',
        name='posts/semana',
    ))
    fig.add_hline(y=threshold, line_dash='dash', line_color='orange',
                  annotation_text=f'µ+2σ = {threshold:.0f}', annotation_position='top left')

    spike_x = [p.start_time for p in spike_periods]
    spike_y = [weekly[p] for p in spike_periods]
    fig.add_trace(go.Scatter(
        x=spike_x, y=spike_y, mode='markers',
        marker=dict(color='yellow', size=10, symbol='star'),
        name='spike (z>2)',
    ))

    for ym, label in EVENTS.items():
        dt = pd.Timestamp(ym + '-01')
        fig.add_vline(x=dt, line_color='gold', line_dash='dot', opacity=0.6,
                      annotation_text=label, annotation_textangle=-90, annotation_font_size=10)

    fig.update_layout(
        title='Actividad semanal IronMarch — picos estadísticos (z>2σ) y eventos externos',
        xaxis_title='Semana', yaxis_title='Posts/semana',
        template='plotly_dark', height=420,
    )
    fig.show()

    print('\nTop semanas pico:')
    spike_info = [(str(weekly.index[i]), int(weekly.values[i]), float(z_scores[i]))
                  for i in range(len(weekly)) if spike_mask[i]]
    spike_info.sort(key=lambda x: x[2], reverse=True)
    for period, count, z in spike_info[:10]:
        print(f'  {period}  posts={count:,}  z={z:.2f}')
else:
    print('Sin columna dateline en posts.')

Actividad semanal:
  Media:   539 posts/semana
  σ:       216 posts/semana
  Umbral (µ+2σ): 970 posts
  Semanas con spike (z>2): 12


/tmp/ipykernel_387018/3393624306.py:18: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  weekly = valid_posts.groupby(valid_posts['dateline'].dt.to_period('W')).size()



Top semanas pico:
  2016-03-28/2016-04-03  posts=1,283  z=3.46
  2012-06-25/2012-07-01  posts=1,235  z=3.23
  2012-03-19/2012-03-25  posts=1,144  z=2.81
  2015-09-21/2015-09-27  posts=1,139  z=2.79
  2017-03-27/2017-04-02  posts=1,101  z=2.61
  2015-10-19/2015-10-25  posts=1,090  z=2.56
  2012-07-02/2012-07-08  posts=1,083  z=2.53
  2015-11-09/2015-11-15  posts=1,083  z=2.53
  2012-03-26/2012-04-01  posts=1,029  z=2.28
  2016-04-04/2016-04-10  posts=1,020  z=2.23


### 4.2 Índice de activación compuesto

El volumen de posts solo mide cuánto se escribe. Pero queremos saber también cuánta
**intensidad emocional** había. Creamos un índice compuesto que combina:

1. **Volumen**: cuántos posts hubo esa semana
2. **Longitud media**: posts más largos = más elaboración = más implicación
3. **Ratio de mayúsculas**: escribir EN MAYÚSCULAS es un indicador de intensidad emocional
4. **Ratio de exclamaciones**: más signos `!` = más activación

Cada componente se normaliza con z-score (para que tengan la misma escala) y promediamos.
Un valor positivo = semana más activa/intensa que la media.

In [9]:
if 'dateline' in posts.columns:
    text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'
    p = posts.dropna(subset=['dateline', text_col]).copy()
    p = p[(p['dateline'].dt.year >= 2011) & (p['dateline'].dt.year <= 2018)]
    p['text_str']   = p[text_col].astype(str)

    # Calcular componentes del índice
    p['nchars']     = p['text_str'].str.len()
    p['cap_ratio']  = p['text_str'].apply(lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1))
    p['excl_ratio'] = p['text_str'].str.count('!') / p['text_str'].str.len().clip(lower=1)
    p['week']       = p['dateline'].dt.to_period('W')

    weekly_act = p.groupby('week').agg(
        n_posts    = ('nchars', 'count'),
        mean_len   = ('nchars', 'mean'),
        cap_ratio  = ('cap_ratio', 'mean'),
        excl_ratio = ('excl_ratio', 'mean'),
    )

    def zscore_col(s):
        return (s - s.mean()) / max(s.std(), 1e-9)

    weekly_act['activation'] = (
        zscore_col(weekly_act['n_posts']) +
        zscore_col(weekly_act['mean_len']) +
        zscore_col(weekly_act['cap_ratio']) +
        zscore_col(weekly_act['excl_ratio'])
    ) / 4

    wx = [period.start_time for period in weekly_act.index]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=wx, y=weekly_act['activation'],
        mode='lines', fill='tozeroy',
        line=dict(color='#FF6B35', width=1.5),
        fillcolor='rgba(255,107,53,0.15)',
        name='activación',
    ))
    fig.add_hline(y=0, line_color='gray', line_dash='dot',
                  annotation_text='media')

    for ym, label in EVENTS.items():
        dt = pd.Timestamp(ym + '-01')
        fig.add_vline(x=dt, line_color='gold', line_dash='dot', opacity=0.6,
                      annotation_text=label, annotation_textangle=-90, annotation_font_size=10)

    top5 = weekly_act['activation'].nlargest(5)
    fig.add_trace(go.Scatter(
        x=[period.start_time for period in top5.index],
        y=top5.values,
        mode='markers', marker=dict(color='yellow', size=10, symbol='star'),
        name='top 5 activación',
    ))

    fig.update_layout(
        title='Índice de activación semanal — IronMarch (z-score compuesto de 4 señales)',
        xaxis_title='Semana', yaxis_title='Índice de activación',
        template='plotly_dark', height=420,
    )
    fig.show()

    print('Top 5 semanas de mayor activación:')
    for period, val in top5.items():
        row = weekly_act.loc[period]
        print(f'  {period.start_time.strftime("%Y-%m-%d")}  '
              f'activación={val:.2f}  posts={int(row["n_posts"])}  len_media={row["mean_len"]:.0f}')

/tmp/ipykernel_387018/618041820.py:11: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  p['week']       = p['dateline'].dt.to_period('W')


Top 5 semanas de mayor activación:
  2015-11-09  activación=2.71  posts=1083  len_media=611
  2012-09-24  activación=1.84  posts=575  len_media=2204
  2012-11-19  activación=1.36  posts=482  len_media=1040
  2012-06-25  activación=1.24  posts=1235  len_media=1269
  2016-02-15  activación=1.18  posts=689  len_media=655


### 4.3 Vocabulario por evento: ¿de qué habla el foro cuando hablan del tema?

El índice de activación mide intensidad emocional (volumen, longitud, MAYÚSCULAS, `!`),
no de qué se habla. Para eso comparamos la frecuencia de una palabra clave en la semana
de un evento contra su frecuencia base (el resto del período 2011-2018).

Los atentados de París son el único evento con pico de **activación** confirmado (sección
4.2). Trump 2016 y Charlottesville 2017 no producen ese pico — pero eso no significa que
el foro no hablara del tema, solo que no se activó emocionalmente por él. Lo comprobamos
calculando la frecuencia de la palabra clave de cada evento en su semana correspondiente.

In [10]:
if 'dateline' in posts.columns:
    text_col_kw = 'pagetext' if 'pagetext' in posts.columns else 'message'
    pk = posts.dropna(subset=['dateline', text_col_kw]).copy()
    pk = pk[(pk['dateline'].dt.year >= 2011) & (pk['dateline'].dt.year <= 2018)]
    pk['text_lower'] = pk[text_col_kw].astype(str).str.lower()
    pk['week'] = pk['dateline'].dt.to_period('W')

    def word_rate(df, word):
        return df['text_lower'].str.count(rf'\b{word}\b').sum() / len(df) if len(df) else 0.0

    # Cada evento se compara en SU PROPIA semana, no todos contra la semana de París
    keyword_events = {
        'Atentados de París (13-nov-2015)': (pd.Period('2015-11-09/2015-11-15', freq='W'), ['paris', 'refugee']),
        'Elección de Trump (8-nov-2016)':    (pd.Period('2016-11-07/2016-11-13', freq='W'), ['trump']),
        'Charlottesville (12-ago-2017)':     (pd.Period('2017-08-07/2017-08-13', freq='W'), ['charlottesville']),
    }

    print(f'{"Evento":<38} {"Palabra":<16} {"Base/post":>10} {"Semana/post":>12} {"Ratio":>8}')
    print('-' * 88)
    for label, (wk, words) in keyword_events.items():
        subset   = pk[pk['week'] == wk]
        baseline = pk[pk['week'] != wk]
        for w in words:
            b = word_rate(baseline, w)
            s = word_rate(subset, w)
            ratio = s / b if b > 0 else float('inf')
            print(f'{label:<38} {w:<16} {b:>10.5f} {s:>12.5f} {ratio:>7.1f}x')

    print()
    print('Activación (sección 4.2) en las mismas semanas:')
    for label, (wk, _) in keyword_events.items():
        if wk in weekly_act.index:
            print(f'  {label:<38} z={weekly_act.loc[wk, "activation"]:.2f}')

Evento                                 Palabra           Base/post  Semana/post    Ratio
----------------------------------------------------------------------------------------


/tmp/ipykernel_387018/1308421516.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  pk['week'] = pk['dateline'].dt.to_period('W')


Atentados de París (13-nov-2015)       paris               0.00371      0.06094    16.4x


Atentados de París (13-nov-2015)       refugee             0.00233      0.00646     2.8x


Elección de Trump (8-nov-2016)         trump               0.01209      0.29981    24.8x


Charlottesville (12-ago-2017)          charlottesville     0.00043      0.02979    69.1x

Activación (sección 4.2) en las mismas semanas:
  Atentados de París (13-nov-2015)       z=2.71
  Elección de Trump (8-nov-2016)         z=0.25
  Charlottesville (12-ago-2017)          z=0.28


## 5. Análisis TF-IDF: términos discriminantes por subforo

**TF-IDF** (Term Frequency × Inverse Document Frequency) es una técnica para encontrar las
palabras que son características de un subforo comparado con el resto.

- **TF** (frecuencia de término): cuántas veces aparece una palabra en los posts de un subforo
- **IDF** (frecuencia inversa de documento): penaliza las palabras que aparecen en todos los
  subforos por igual (como "el", "de", "que") y amplifica las que son exclusivas de uno

El resultado nos dice: **¿qué vocabulario es característico de cada sección del foro?**
Esto es una forma de mapear el ideario sin leer miles de posts.

**Hiperparámetros del vectorizador:** `max_features=500` limita el vocabulario a las 500
palabras más informativas; `min_df=2` ignora palabras que solo aparecen en 1 subforo
(probable ruido).

In [11]:
import re

# Necesitamos forumid en posts — usar el join con threads
if 'forumid' not in posts.columns and 'forumid' in threads.columns:
    thread_to_forum = threads.set_index('threadid')['forumid'].to_dict()
    posts = posts.copy()
    posts['forumid'] = posts['threadid'].astype(str).map(thread_to_forum)

text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'

if 'forumid' in posts.columns:
    def clean_for_tfidf(text: str) -> str:
        """Limpiar texto para TF-IDF: sin HTML, solo letras en minúsculas."""
        text = re.sub(r'<[^>]+>', ' ', str(text))
        text = re.sub(r'https?://\S+', ' ', text)
        return re.sub(r'[^a-zA-Z\s]', ' ', text).lower()

    # Concatenar todos los posts de cada subforo como un solo documento
    forum_corpus = (
        posts.dropna(subset=['forumid', text_col])
        .groupby('forumid')[text_col]
        .apply(lambda texts: ' '.join(texts.astype(str).tolist()))
        .reset_index()
    )
    forum_corpus['text_clean'] = forum_corpus[text_col].apply(clean_for_tfidf)

    # Obtener nombres de foros
    if 'name' in forums.columns and 'forumid' in forums.columns:
        fid_to_name = dict(zip(forums['forumid'].astype(str), forums['name']))
        forum_corpus['forum_name'] = forum_corpus['forumid'].astype(str).map(fid_to_name).fillna('desconocido')
    else:
        forum_corpus['forum_name'] = forum_corpus['forumid'].astype(str)

    # Aplicar TF-IDF (hiperparámetros explicados en el markdown arriba)
    vectorizer = TfidfVectorizer(
        stop_words='english', max_features=500, min_df=2, ngram_range=(1, 2)
    )
    tfidf_matrix = vectorizer.fit_transform(forum_corpus['text_clean'])
    feature_names = vectorizer.get_feature_names_out()

    print(f'Subforos analizados: {len(forum_corpus)}')
    print(f'Vocabulario TF-IDF: {len(feature_names):,} términos')
    print()

    # Top 8 términos por subforo
    tfidf_array = tfidf_matrix.toarray()
    top_n = 8

    print('Top términos TF-IDF por subforo (top 10 subforos más activos):')
    post_counts_by_forum = posts.groupby('forumid').size()
    top_forums = post_counts_by_forum.nlargest(10).index.astype(str).tolist()

    for _, row in forum_corpus.iterrows():
        if str(row['forumid']) not in top_forums:
            continue
        idx = forum_corpus.index[forum_corpus['forumid'] == row['forumid']][0]
        scores = tfidf_array[idx]
        top_idx = scores.argsort()[::-1][:top_n]
        top_terms = [feature_names[i] for i in top_idx]
        print(f'  [{row["forum_name"][:40]}]: {" | ".join(top_terms)}')
else:
    print('No hay forumid disponible. Revisar el pipeline de datos.')

Subforos analizados: 126
Vocabulario TF-IDF: 500 términos

Top términos TF-IDF por subforo (top 10 subforos más activos):
  [Images]: said | just | ago | gt | like | png | emoticons | hours ago
  [Introductions]: index php | php | index | rel | color | underline background | color transparent | decoration underline
  [General Discussions]: just | like | people | don | said | think | really | time


  [Wastelands]: just | said | like | people | don | ago | white | think
  [Against the Modern World]: people | like | just | don | think | said | women | know
  [The Showers]: just | said | like | people | don | fucking | think | fascist
  [Cancer]: just | like | said | people | don | think | shit | gt
  [Fascist News]: people | like | said | just | right | don | white | think
  [Shill News]: people | just | like | said | don | think | jews | white
  [Iron Lounge]: just | like | people | said | don | ago | know | time


## 6. Visualización de la red de PMs

La red de mensajes privados tiene la misma lógica visual que la red pública, pero el
**color** ahora representa el **in-degree** (cuántos mensajes privados recibe ese usuario).
Un in-degree alto en la red de PMs indica que ese usuario es muy buscado como interlocutor privado.

In [12]:
if G_pm.number_of_nodes() > 0 and 'btw_pm' in dir():
    G_pm_und = G_pm.to_undirected()
    top_pm_nodes = sorted(G_pm_und.degree(), key=lambda x: x[1], reverse=True)[:150]
    top_pm_set   = {n for n, _ in top_pm_nodes}
    G_pm_viz     = G_pm_und.subgraph(top_pm_set).copy()

    pos = nx.spring_layout(G_pm_viz, k=0.8, iterations=60, seed=42)

    edge_x, edge_y = [], []
    for u, v in G_pm_viz.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.5, color='rgba(150,150,150,0.25)'),
        hoverinfo='none',
    )

    node_indeg  = [in_deg.get(n, 0) for n in G_pm_viz.nodes()]
    node_btw_pm = [btw_pm.get(n, 0) for n in G_pm_viz.nodes()]
    node_names  = [uid_to_name.get(n, f'uid:{n}') for n in G_pm_viz.nodes()]
    top8_pm     = {uid for uid, _ in top_pm_brokers[:8]}

    node_trace = go.Scatter(
        x=[pos[n][0] for n in G_pm_viz.nodes()],
        y=[pos[n][1] for n in G_pm_viz.nodes()],
        mode='markers+text',
        marker=dict(
            size=[b * 80 + 6 for b in node_btw_pm],
            color=node_indeg,
            colorscale='Plasma',
            colorbar=dict(title='PMs recibidos'),
            line=dict(width=0.5, color='white'),
        ),
        text=[name if n in top8_pm else '' for n, name in zip(G_pm_viz.nodes(), node_names)],
        textposition='top center',
        textfont=dict(size=9, color='white'),
        hovertext=[
            f'{name}<br>PMs recibidos={in_deg.get(n,0)}<br>btw={btw_pm.get(n,0):.4f}'
            for n, name in zip(G_pm_viz.nodes(), node_names)
        ],
        hoverinfo='text',
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title='Red de mensajes privados — IronMarch (top 150, hover para detalles)',
            template='plotly_dark', showlegend=False,
            width=1000, height=750,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=50, b=20),
        )
    )
    fig.show()
else:
    print('Grafo de PMs vacío o betweenness no calculado.')

In [13]:
# Guardar señales estructurales para el notebook de síntesis
if G_public.number_of_nodes() > 0 and 'btw_cent' in dir():
    structural_signals = pd.DataFrame([
        {
            'userid': uid,
            'username': uid_to_name.get(uid, uid),
            'degree_centrality': degree_cent.get(uid, 0),
            'betweenness_public': btw_cent.get(uid, 0),
            'betweenness_pm': btw_pm.get(uid, 0) if 'btw_pm' in dir() else None,
            'pm_in_degree': in_deg.get(uid, 0) if 'in_deg' in dir() else None,
            'community_id': partition.get(uid) if 'partition' in dir() else None,
        }
        for uid in set(btw_cent.keys())
    ])
    structural_signals = structural_signals.sort_values('betweenness_public', ascending=False)

    out_path = IM_RESULTS / 'structural_signals.parquet'
    structural_signals.to_parquet(out_path, index=False)
    print(f'Señales estructurales guardadas: {out_path}')
    print(f'  {len(structural_signals):,} usuarios con datos de centralidad')
    print(f'\nSiguiente paso: 03_analisis_semantico.ipynb')

Señales estructurales guardadas: /home/miguel/csbc26/results/ironmarch/structural_signals.parquet
  1,153 usuarios con datos de centralidad

Siguiente paso: 03_analisis_semantico.ipynb
